# ZuCo 2.0 EEG Data Extraction & Visualization Pipeline
This notebook provides a pure Python pipeline to read ZuCo 2.0 `.mat` files (v7.3 format), extract the EEG data, and visualize the EEG mappings at both the **sentence level** and **word level**.

### Prerequisites
Make sure you have the required libraries installed:
```bash
pip install h5py numpy matplotlib scipy
```

### Download ZuCo 2.0 Dataset
The following cell contains the download script to automatically download the ZuCo 2.0 dataset from OSF (approx 120GB). It supports resuming if interrupted.

In [ ]:
import os
import sys

try:
    import requests
    from tqdm import tqdm
except ImportError:
    print("Please install required packages before running:")
    print("pip install requests tqdm")
    sys.exit(1)

# OSF Node ID for ZuCo 2.0
NODE_ID = "2urht"
BASE_URL = f"https://api.osf.io/v2/nodes/{NODE_ID}/files/osfstorage/"

# Resolve dataset directory (../dataset/zuco2)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

import time


def get_osf_data(api_url, max_retries=5):
    """Fetch folder metadata from OSF API with retries, handling pagination."""
    all_data = []
    current_url = api_url

    while current_url:
        for attempt in range(max_retries):
            try:
                response = requests.get(current_url, timeout=30)
                response.raise_for_status()
                json_data = response.json()
                all_data.extend(json_data["data"])

                # Check for next page
                links = json_data.get("links", {})
                current_url = links.get("next")
                break  # Break retry loop if successful
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    print(
                        f"Network error while fetching metadata: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                    )
                    time.sleep(5)
                else:
                    raise
    return all_data


def download_file_resumable(url, destination, max_retries=5):
    """Downloads a file with resume support and retries."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temp_destination = destination + ".tmp"

    for attempt in range(max_retries):
        file_size = 0
        if os.path.exists(destination):
            print(
                f"File {os.path.basename(destination)} already fully downloaded. Skipping."
            )
            return True

        if os.path.exists(temp_destination):
            file_size = os.path.getsize(temp_destination)

        headers = {"Range": f"bytes={file_size}-"} if file_size > 0 else {}

        try:
            response = requests.get(
                url, headers=headers, stream=True, allow_redirects=True, timeout=30
            )

            # 416 Range Not Satisfiable means we requested a range past the end of the file
            if response.status_code == 416:
                os.rename(temp_destination, destination)
                print(f"File {os.path.basename(destination)} already fully downloaded.")
                return True

            if response.status_code not in [200, 206]:
                print(f"Failed to download {url}. Status code: {response.status_code}")
                return False

            if file_size > 0 and response.status_code == 200:
                print("Server doesn't support resume. Restarting download...")
                file_size = 0
                mode = "wb"
            else:
                mode = "ab"

            total_size = int(response.headers.get("content-length", 0)) + file_size

            with (
                open(temp_destination, mode) as f,
                tqdm(
                    desc=os.path.basename(destination),
                    total=total_size,
                    initial=file_size,
                    unit="iB",
                    unit_scale=True,
                    unit_divisor=1024,
                ) as bar,
            ):
                for chunk in response.iter_content(chunk_size=8192 * 4):
                    if chunk:
                        size = f.write(chunk)
                        bar.update(size)

            # Verify the file is complete
            if total_size == 0 or os.path.getsize(temp_destination) >= total_size:
                os.rename(temp_destination, destination)
                return True
            else:
                print(f"Download incomplete for {os.path.basename(destination)}")
                # Will retry in the next loop iteration

        except Exception as e:
            if attempt < max_retries - 1:
                print(
                    f"\nError downloading {os.path.basename(destination)}: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                )
                time.sleep(5)
            else:
                print(
                    f"\nFailed to download {os.path.basename(destination)} after {max_retries} attempts: {e}"
                )
                return False


def traverse_and_download(api_url, current_path):
    """Recursively traverses OSF folders and downloads files sequentially."""
    print(f"Fetching listing for {os.path.relpath(current_path, PROJECT_ROOT)} ...")
    items = get_osf_data(api_url)

    for item in items:
        kind = item["attributes"]["kind"]
        name = item["attributes"]["name"]

        if kind == "folder":
            next_url = item["relationships"]["files"]["links"]["related"]["href"]
            next_path = os.path.join(current_path, name)
            success = traverse_and_download(next_url, next_path)
            if not success:
                return False
        elif kind == "file":
            download_url = item["links"]["download"]
            file_path = os.path.join(current_path, name)

            # Download file sequentially, stopping if one fails
            success = download_file_resumable(download_url, file_path)
            if not success:
                print(f"Stopping download process because {name} failed.")
                return False

    return True


if __name__ == "__main__":
    print(f"Starting download of ZuCo 2.0 to: {DATASET_DIR}")
    print("-" * 50)
    os.makedirs(DATASET_DIR, exist_ok=True)

    success = traverse_and_download(BASE_URL, DATASET_DIR)

    if success:
        print("-" * 50)
        print("Dataset download completed successfully!")
    else:
        print("-" * 50)
        print("Dataset download interrupted. Run the script again to resume.")

In [ ]:
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np

# Setup paths
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

# Output directories for visualizations
WORD_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "word eeg mapping")
SENTENCE_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "sentence eeg mapping")

os.makedirs(WORD_MAPPING_DIR, exist_ok=True)
os.makedirs(SENTENCE_MAPPING_DIR, exist_ok=True)

print(f"Data directory: {DATASET_DIR}")
print(f"Word mapping output: {WORD_MAPPING_DIR}")
print(f"Sentence mapping output: {SENTENCE_MAPPING_DIR}")

### Helper Functions
Functions to extract strings from `h5py` references and plot the EEG graphs.

In [ ]:
def get_string(f, ref):
    """Extracts string from h5py object reference."""
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except:
        return "Unknown"


def plot_eeg(eeg_data, title, filename):
    """
    Plots all EEG channels on a single graph and saves the image.
    eeg_data shape is usually (channels, time) or (time, channels).
    We assume the longer dimension is time.
    """
    if eeg_data is None or eeg_data.size == 0:
        return

    eeg_data = np.array(eeg_data)
    # Ensure shape is (channels, time)
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T

    num_channels, time_points = eeg_data.shape

    plt.figure(figsize=(15, 8))
    for i in range(num_channels):
        # Offset each channel to visualize them stacked
        offset = i * (np.max(eeg_data) - np.min(eeg_data)) * 0.5
        plt.plot(eeg_data[i, :] + offset, linewidth=0.5, alpha=0.8)

    plt.title(title)
    plt.xlabel("Time points")
    plt.ylabel("EEG Channels (Offset)")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    plt.close()

### Extraction and Visualization Loop
This loop iterates through the downloaded `.mat` files, extracts sentence and word data, and plots them.

In [ ]:
# Find all .mat files in the dataset directory
mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "*.mat"), recursive=True)
print(f"Found {len(mat_files)} .mat files.")

if not mat_files:
    print("No .mat files found. Make sure the download script has finished running.")

# We limit to the first file for demonstration, or loop through all
for mat_file in mat_files[:1]:  # Remove [:1] to process all files
    subject_name = os.path.basename(mat_file).replace(".mat", "")
    print(f"Processing {subject_name}...")

    try:
        with h5py.File(mat_file, "r") as f:
            # ZuCo data is typically under 'sentenceData'
            if "sentenceData" not in f:
                print(
                    f"'sentenceData' not found in {subject_name}. Available keys: {list(f.keys())}"
                )
                continue

            sentence_data = f["sentenceData"]

            # sentence_data is an array of object references
            for i, sent_ref in enumerate(sentence_data[0, :]):
                sent_obj = f[sent_ref]

                # 1. Extract Sentence
                sent_text = "sentence_text"
                if "content" in sent_obj:
                    sent_text = get_string(f, sent_obj["content"][0, 0])
                else:
                    sent_text = f"Sentence_{i + 1}"

                safe_sent_name = "".join(
                    [c if c.isalnum() else "_" for c in sent_text]
                )[:50]

                # 2. Sentence EEG Mapping
                # ZuCo has different EEG features (e.g., 'mean_t1', 'mean_t2', 'mean_a1' or 'rawData')
                # We will look for raw EEG or a specific feature band
                eeg_sent = None
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sent_obj:
                        eeg_sent = sent_obj[key][:]
                        break

                if eeg_sent is not None:
                    out_name = os.path.join(
                        SENTENCE_MAPPING_DIR,
                        f"{subject_name}_sent_{i}_{safe_sent_name}.png",
                    )
                    plot_eeg(eeg_sent, f"Sentence EEG: {sent_text}", out_name)

                # 3. Word EEG Mapping
                if "word" in sent_obj:
                    words_refs = sent_obj["word"]

                    # check if word is an array of references
                    try:
                        num_words = words_refs.shape[0] * words_refs.shape[1]
                        words_flat = words_refs[:].flatten()

                        for w_idx, w_ref in enumerate(words_flat):
                            # Skip if empty reference
                            if not w_ref:
                                continue

                            word_obj = f[w_ref]

                            word_text = f"word_{w_idx}"
                            if "content" in word_obj:
                                word_text = get_string(f, word_obj["content"][0, 0])

                            safe_word_name = "".join(
                                [c if c.isalnum() else "_" for c in word_text]
                            )

                            eeg_word = None
                            for key in ["rawData", "mean_t1", "mean_t2"]:
                                if key in word_obj:
                                    eeg_word = word_obj[key][:]
                                    break

                            if eeg_word is not None:
                                out_name = os.path.join(
                                    WORD_MAPPING_DIR,
                                    f"{subject_name}_sent_{i}_word_{w_idx}_{safe_word_name}.png",
                                )
                                plot_eeg(eeg_word, f"Word EEG: {word_text}", out_name)
                    except Exception as e:
                        print(f"Could not parse words for sentence {i}: {e}")

        print(f"Finished processing {subject_name}.")
    except Exception as e:
        print(f"Error processing {mat_file}: {e}")

print("Visualization pipeline completed. Check the 'dataset' subfolders for images.")